# Lab 2 — The toolbox for sequences: recurrent networks

Time series, text, audio, sensor signals: many data are **sequences**, where the order matters and the length can
vary. Recurrent networks process a sequence one step at a time, carrying along a **hidden state** $h_t$ that acts as
a memory of what they have seen so far.

In this notebook:
1. The shape of sequence tensors
2. `nn.RNN` — the basic recurrent layer (and what it computes)
3. `nn.LSTM` and `nn.GRU`
4. Which output do I use? Many-to-one vs many-to-many
5. From a time series to a `DataLoader` of sequences
6. A complete example: forecasting with an LSTM

Afterwards, in `rnn_timeseries.py`, you will build your own network to forecast a real time series.

In [ ]:
import math

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

torch.manual_seed(0)

---
## 1. The shape of sequence tensors

A batch of sequences is a 3D tensor. With `batch_first=True` (which we will **always** use) the format is:

$$(\text{batch},\ \text{seq\_len},\ \text{features})$$

For example, 32 windows of 30 days of temperature: `(32, 30, 1)`. If for each day we also had humidity and pressure:
`(32, 30, 3)`.

⚠️ The PyTorch default is `batch_first=False`, i.e. `(seq_len, batch, features)`: forgetting `batch_first=True` is a
classic bug that does **not** raise an error, it just silently mixes up batch and time!

---
## 2. `nn.RNN` — the basic recurrent layer

At each time step $t$ the RNN combines the current input $x_t$ with the previous hidden state $h_{t-1}$:

$$h_t = \tanh\left(W_{ih}\, x_t + b_{ih} + W_{hh}\, h_{t-1} + b_{hh}\right)$$

The **same weights** are used at every step (just like a convolution reuses the same filter at every position).
Main arguments: `input_size` (features per step) and `hidden_size` (size of $h_t$).

In [ ]:
rnn = nn.RNN(input_size=3, hidden_size=16, batch_first=True)

x = torch.randn(4, 10, 3)     # batch=4, seq_len=10, features=3
out, h_n = rnn(x)

print("input: ", tuple(x.shape))
print("out:   ", tuple(out.shape), " -> all the h_t, one per step: (batch, seq_len, hidden)")
print("h_n:   ", tuple(h_n.shape), " -> only the last h_t: (1, batch, hidden)")

It returns two things:
- `out`: the **sequence** of all hidden states $h_1, \dots, h_L$, one per time step → shape `(batch, seq_len, hidden)`;
- `h_n`: only the **last** one, $h_L$ → shape `(1, batch, hidden)` (the leading `1` is the number of layers).
  ⚠️ `h_n` is *not* batch-first, even with `batch_first=True`.

So `h_n` is just a shortcut: `h_n[0]` is the same as `out[:, -1]`. In practice we will always use `out`.

Let's convince ourselves that the formula above is really all an RNN does, by rewriting it with a `for` loop over time
(→ tensors §8, matrix product):

In [ ]:
rnn = nn.RNN(input_size=1, hidden_size=4, batch_first=True)
x = torch.randn(1, 5, 1)
out, h_n = rnn(x)

h = torch.zeros(1, 4)                     # initial hidden state h_0 = 0
for t in range(x.shape[1]):
    x_t = x[:, t]                         # input at time t, shape (1, 1)
    h = torch.tanh(x_t @ rnn.weight_ih_l0.T + rnn.bias_ih_l0 + h @ rnn.weight_hh_l0.T + rnn.bias_hh_l0)

print("Manual loop == nn.RNN?", torch.allclose(h, h_n[0]), torch.allclose(h, out[:, -1]))

---
## 3. `nn.LSTM` and `nn.GRU`

The basic RNN struggles to remember information for many steps: during backpropagation the gradient is multiplied
by the same matrix at every step and tends to vanish (→ tensors §13.4, vanishing gradient).

**LSTM** (Long Short-Term Memory) adds a second state, the **cell state** $c_t$, and three **gates** (input, forget,
output): small learned "valves" with a sigmoid that decide what to write into memory, what to forget and what to
output. **GRU** (Gated Recurrent Unit) is a lighter version with two gates and no cell state.

The interface is the same as `nn.RNN`; the only difference is that the LSTM returns a tuple `(h_n, c_n)`:

In [ ]:
lstm = nn.LSTM(input_size=3, hidden_size=16, batch_first=True)
gru = nn.GRU(input_size=3, hidden_size=16, batch_first=True)

x = torch.randn(4, 10, 3)
out, (h_n, c_n) = lstm(x)          # LSTM: hidden state AND cell state
print("LSTM  out:", tuple(out.shape), " h_n:", tuple(h_n.shape), " c_n:", tuple(c_n.shape))

out, h_n = gru(x)                  # GRU: like the RNN
print("GRU   out:", tuple(out.shape), " h_n:", tuple(h_n.shape))

The gates cost parameters: the LSTM has 4 sets of weights (3 gates + the candidate), the GRU 3.

In [ ]:
def n_params(m):
    return sum(p.numel() for p in m.parameters())

for name, cls in [("RNN", nn.RNN), ("GRU", nn.GRU), ("LSTM", nn.LSTM)]:
    print(f"{name:5s} {n_params(cls(input_size=3, hidden_size=16)):>5,} parameters")

💡 In practice: LSTM and GRU perform similarly; GRU is a bit faster. The basic RNN is rarely used.

---
## 4. Which output do I use?

It depends on the task:
- **many-to-one** (one prediction per sequence, e.g. "tomorrow's temperature" or "is this review positive?"):
  take the hidden state of the **last step**, `out[:, -1, :]`, and feed it to a `nn.Linear`;
- **many-to-many** (one prediction per step, e.g. tagging each word of a sentence): apply the `nn.Linear` to
  **all** of `out`. `nn.Linear` works on the last dimension, so it is applied to every step automatically.

In [ ]:
out, (h_n, c_n) = lstm(x)
head = nn.Linear(16, 1)

many_to_one = head(out[:, -1, :])
many_to_many = head(out)
print("many-to-one: ", tuple(many_to_one.shape))    # (batch, 1)
print("many-to-many:", tuple(many_to_many.shape))   # (batch, seq_len, 1)

# The last step of `out` is the final hidden state h_n[0]
print("\nout[:, -1] is the same as h_n[0]?", torch.allclose(out[:, -1], h_n[0]))

---
## 5. From a time series to a `DataLoader` of sequences

A time series is a single long sequence of values. To train a network to **forecast** it, we cut it into many
samples with a **sliding window**: the input is a window of `window` consecutive values, the target is the value
that comes right after.

```
series:  v0 v1 v2 v3 v4 v5 v6 v7 ...
sample 0: [v0 v1 v2 v3] -> v4
sample 1:    [v1 v2 v3 v4] -> v5
sample 2:       [v2 v3 v4 v5] -> v6
```

Let's create a toy series: two overlapping sine waves plus noise.

In [ ]:
t = torch.arange(1500, dtype=torch.float32)
series = torch.sin(2 * math.pi * t / 50) + 0.5 * torch.sin(2 * math.pi * t / 120) + 0.1 * torch.randn(len(t))

plt.figure(figsize=(12, 3))
plt.plot(t[:500], series[:500])
plt.title("Toy time series (first 500 steps)")
plt.show()

### Train/test split and normalization

⚠️ Two important rules for time series:
1. **Split chronologically**: train on the past, test on the future. A random split would let the model "peek" at
   values right next to the test points.
2. **Normalize with the training statistics only**, then apply the same mean/std to the test set (the test set
   represents the unknown future).

In [ ]:
split = int(0.8 * len(series))
train_series, test_series = series[:split], series[split:]

mean, std = train_series.mean(), train_series.std()
train_series = (train_series - mean) / std
test_series = (test_series - mean) / std
print(f"Train: {len(train_series)} steps, test: {len(test_series)} steps")

### The `Dataset`

The sliding window fits naturally into a custom `Dataset` (→ training §2.3): `__getitem__(i)` returns the window
starting at `i` and the value that follows it. Note the `unsqueeze(-1)`: it adds the *features* dimension, because
the RNN wants `(seq_len, features)` for each sample.

In [ ]:
class SlidingWindowDataset(Dataset):
    def __init__(self, series, window):
        self.series = series
        self.window = window

    def __len__(self):
        return len(self.series) - self.window

    def __getitem__(self, i):
        x = self.series[i : i + self.window].unsqueeze(-1)      # (window, 1)
        y = self.series[i + self.window].unsqueeze(-1)          # (1,)
        return x, y


window = 60
train_ds = SlidingWindowDataset(train_series, window)
test_ds = SlidingWindowDataset(test_series, window)
print(f"Training samples: {len(train_ds)}, test samples: {len(test_ds)}")

x0, y0 = train_ds[0]
print(f"One sample: x {tuple(x0.shape)}, y {tuple(y0.shape)}")

plt.figure(figsize=(9, 3))
plt.plot(range(window), x0.squeeze(), "o-", ms=3, label="input window")
plt.plot(window, y0, "r*", ms=14, label="target")
plt.legend()
plt.title("One training sample")
plt.show()

### The `DataLoader`

Each window is an independent sample, so we **can** shuffle the training windows (the order *inside* each window is
preserved). The `DataLoader` stacks them into batches with exactly the `(batch, seq_len, features)` shape the RNN
expects.

In [ ]:
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

xb, yb = next(iter(train_loader))
print(f"Batch: x {tuple(xb.shape)}  (batch, seq_len, features),  y {tuple(yb.shape)}")

---
## 6. A complete example: forecasting with an LSTM

A many-to-one model: the LSTM reads the window, we take the hidden state of the last step and a linear layer turns it
into the prediction of the next value.

In [ ]:
class Forecaster(nn.Module):
    def __init__(self, hidden_size=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):                 # x: (batch, window, 1)
        out, _ = self.lstm(x)             # out: (batch, window, hidden)
        return self.head(out[:, -1, :])   # last step -> (batch, 1)


model = Forecaster()
print(model)
print(f"Parameters: {n_params(model):,}")

The training loop is the usual one (→ training §4.4); since this is regression, the loss is the MSE.

In [ ]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

for epoch in range(10):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        loss = loss_fn(model(xb), yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    with torch.no_grad():
        test_loss = sum(loss_fn(model(xb), yb).item() for xb, yb in test_loader) / len(test_loader)
    print(f"epoch {epoch + 1:2d}: train MSE {train_loss / len(train_loader):.4f} | test MSE {test_loss:.4f}")

Let's compare the predictions with the real test series. Each prediction uses only the 60 **real** values before it
(one-step-ahead forecasting).

In [ ]:
model.eval()
with torch.no_grad():
    preds = torch.cat([model(xb) for xb, _ in test_loader]).squeeze()
targets = test_series[window:]

plt.figure(figsize=(12, 4))
plt.plot(targets, label="true")
plt.plot(preds, label="LSTM prediction")
plt.legend()
plt.title("One-step-ahead forecast on the test set")
plt.show()

💡 The noise (std 0.1, i.e. ~0.13 after normalization) cannot be predicted: the best possible test MSE is about
$0.13^2 \approx 0.02$. The model is not far from it!

---
## Summary

| Tool | Input | Returns |
|---|---|---|
| `nn.RNN(input_size, hidden_size, batch_first=True)` | `(B, L, F)` | `out (B, L, H)`, `h_n (1, B, H)` |
| `nn.LSTM(...)` | `(B, L, F)` | `out (B, L, H)`, `(h_n, c_n)` |
| `nn.GRU(...)` | `(B, L, F)` | `out (B, L, H)`, `h_n` |
| many-to-one | | `nn.Linear(H, out)` applied to `out[:, -1, :]` |
| many-to-many | | `nn.Linear(H, out)` applied to `out` |

Remember: `out` = **sequence of all $h_t$**; `h_n` = **only the last $h_t$**.

Recipe for time series: **chronological split → normalize with train stats → sliding windows → DataLoader**.

👉 **Now it's your turn:** open `rnn_timeseries.py` and build your network in the `TODO`.